# Import & GPU Utils

In [7]:
import numpy as np
import pandas as pd
import warnings

warnings.filterwarnings("ignore")

import torch
import random
import os

import torch.nn as nn

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

USE_AMP = (DEVICE.type == "cuda" and os.environ.get("TSF_USE_AMP", "1") != "0")
USE_TORCH_COMPILE = (os.environ.get("TSF_TORCH_COMPILE", "0") == "1")

In [ ]:
TARGET_COL = "max_temperature"
FEATURE_COLS = [
    "max_temperature",
    "latitude",
    "longitude",
    "elevation",
    "sin_doy",
    "cos_doy",
    "solar_declination",
    "dmi east",
    "nino anom 3.4",
]
INPUT_LEN = 14
HORIZON = 7
STRIDE = 2


LOCAL_TEMPORAL_COLS = ["max_temperature", "sin_doy", "cos_doy", "solar_declination"]
GEO_COLS = ["latitude", "longitude", "elevation"]
CLIMATE_COLS = ["nino anom 3.4", "dmi east"]


In [2]:
_CPU_CORES = os.cpu_count() or 4
# Threadripper 3960X = 24 cores (48 threads). For NumPy/torch this is a good default.
DEFAULT_CPU_THREADS = min(24, _CPU_CORES)

# Only set if user hasn't already configured these in their environment.
os.environ.setdefault("OMP_NUM_THREADS", str(DEFAULT_CPU_THREADS))
os.environ.setdefault("MKL_NUM_THREADS", str(DEFAULT_CPU_THREADS))


import random
import torch



torch.set_default_dtype(torch.float32)
torch.set_num_threads(DEFAULT_CPU_THREADS)
torch.set_num_interop_threads(max(1, min(4, DEFAULT_CPU_THREADS // 2)))

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

if DEVICE.type == "cuda":
    # Ampere GPU (3090 Ti): TF32 is a big speedup for matmul/conv with minimal impact for this task.
    print(f"Training With: {DEVICE}")
    _use_tf32 = (os.environ.get("TSF_USE_TF32", "1") != "0")
    if _use_tf32:
        try:
            torch.backends.cuda.matmul.allow_tf32 = True
            torch.backends.cudnn.allow_tf32 = True
        except Exception:
            pass
    torch.backends.cudnn.benchmark = True

USE_AMP = (DEVICE.type == "cuda" and os.environ.get("TSF_USE_AMP", "1") != "0")
USE_TORCH_COMPILE = (os.environ.get("TSF_TORCH_COMPILE", "0") == "1")


def _maybe_compile(model: torch.nn.Module) -> torch.nn.Module:
    if not USE_TORCH_COMPILE:
        return model
    try:
        return torch.compile(model)  # PyTorch 2.x
    except Exception:
        return model


def _dataloader_kwargs(shuffle: bool, drop_last: bool = False):
    """
    Sensible DataLoader defaults for this workstation.
    - Use CPU workers to overlap preprocessing/host->device transfers.
    - Use pin_memory for faster H2D copies when using CUDA.
    """
    num_workers = 0
    if DEVICE.type == "cuda":
        # Windows multiprocessing can be heavier; keep this conservative but non-zero.
        num_workers = min(8, max(2, DEFAULT_CPU_THREADS // 3))

    kwargs = dict(
        shuffle=bool(shuffle),
        num_workers=int(num_workers),
        pin_memory=(DEVICE.type == "cuda"),
        drop_last=bool(drop_last),
    )
    if kwargs["num_workers"] > 0:
        kwargs["persistent_workers"] = True
        kwargs["prefetch_factor"] = 2
    return kwargs


def set_seed(seed: int = 42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)

Training With: cuda


In [25]:
results_rows = []
def log_row(**kwargs):
    row = {
            "target_region": "papua",
            "source_region": "java",
            "TARGET_COL": TARGET_COL,
            "INPUT_LEN": INPUT_LEN,
            "HORIZON": HORIZON,
            "STRIDE": STRIDE,
            "n_features": len(FEATURE_COLS),
        }
    row.update(kwargs)
    results_rows.append(row)

# Windowing

In [ ]:
def load_and_split(data_path: str):
    """Load merged_dataset.csv and split by time. Return (train, val, test) DataFrames."""
    df = pd.read_csv(data_path, dtype={"location_id": str})
    # Normalize column names (e.g. if CSV uses "temperature_2m_max (°C)" or "DMI EAST")
    col_map = {}
    for c in list(df.columns):
        c_lower = c.strip().lower()
        if "temperature" in c_lower and "max" in c_lower and c != "max_temperature":
            col_map[c] = "max_temperature"
        if "dmi" in c_lower and "east" in c_lower and c != "dmi east":
            col_map[c] = "dmi east"
        if "nino" in c_lower and "3.4" in c_lower and c != "nino anom 3.4":
            col_map[c] = "nino anom 3.4"
    df = df.rename(columns=col_map)
    df["time"] = pd.to_datetime(df["time"])
    df = df.sort_values(["location_id", "time"]).reset_index(drop=True)

    train = df[(df["time"] >= "2005-01-01") & (df["time"] <= "2018-12-31")]
    val   = df[(df["time"] >= "2019-01-01") & (df["time"] <= "2021-12-31")]
    test  = df[(df["time"] >= "2022-01-01") & (df["time"] <= "2025-05-01")]

    return train, val, test


def build_windows_one_location(times, values, input_len, horizon, stride):
    """Build (X, y) windows for one location. values: (T, n_features)."""
    T = len(times)
    # if the station does not have enough values to form a full window # we skip
    if T < input_len + horizon:
        return None, None
    X_list, y_list = [], []
    for start in range(0, T - input_len - horizon + 1, stride):
        end_in = start + input_len
        end_out = end_in + horizon
        X_list.append(values[start:end_in])
        y_list.append(values[end_in:end_out, 0])
    if not X_list:
        return None, None
    return np.array(X_list, dtype=np.float32), np.array(y_list, dtype=np.float32)


def build_windows_temp_transformer_one_location(times, values, input_len, horizon, stride):
    """Build windows for `TemperatureTransformer`.

    Returns:
      - z_past: (input_len, 1) temperature history
      - x_cov_past: (input_len, d_cov) past covariates (features 1:)
      - x_cov_future: (horizon, d_cov) future covariates (features 1:)
      - y: (horizon,) future temperature target (feature 0)

    `values` must be shaped (T, n_features) where feature 0 is temperature.
    """
    T = len(times)
    if T < input_len + horizon:
        return None, None, None, None

    z_past_list, x_cov_past_list, x_cov_future_list, y_list = [], [], [], []

    for start in range(0, T - input_len - horizon + 1, stride):
        end_in = start + input_len
        end_out = end_in + horizon

        past = values[start:end_in]       # (input_len, n_features)
        future = values[end_in:end_out]  # (horizon, n_features)

        z_past_list.append(past[:, :1])
        x_cov_past_list.append(past[:, 1:])
        x_cov_future_list.append(future[:, 1:])
        y_list.append(future[:, 0])

    if not z_past_list:
        return None, None, None, None

    return (
        np.array(z_past_list, dtype=np.float32),
        np.array(x_cov_past_list, dtype=np.float32),
        np.array(x_cov_future_list, dtype=np.float32),
        np.array(y_list, dtype=np.float32),
    )


def build_all_windows(train, val, test):
    """Build X, y for train/val/test. No cross location_id, no cross split boundary."""
    def run_split(split_df):
        X_list, y_list = [],[]
        grp = grp.sort_values("time")
        mat = grp[FEATURE_COLS].values.astype(np.float32)
        times = grp["time"].values
        X, y = build_windows_one_location(times, mat, INPUT_LEN, HORIZON, STRIDE)
        if not X is not None:
            X_list.append(X)
            y_list.append(y)
        if not X_list:
            return np.zeros((0, INPUT_LEN, len(FEATURE_COLS)), dtype=np.float32), np.zeros((0, INPUT_LEN, len(FEATURE_COLS)), dtype=np.float32)
        return np.concatenate(X_list, axis=0), np.concatenate(y_list, axis=0)
    X_train, y_train = run_split(train)
    X_val,   y_val   = run_split(val)
    X_test,  y_test  = run_split(test)
    return X_train, y_train, X_val, y_val, X_test, y_test




In [4]:
def build_windows(split_df: pd.DataFrame):
    """Build(X, y) for a single split dataframe (across all location_id)"""
    X_list, y_list = [], []
    for _, grp in split_df.groupby("location_id"):
        grp = grp.sort_values("time")
        mat = grp[FEATURE_COLS].values.astype(np.float32)
        times = grp['time'].values
        X, y = build_windows_one_location(times, mat, INPUT_LEN, HORIZON, STRIDE)
        if X is not None:
            X_list.append(X)
            y_list.append(y)
    if not X_list:
        return (
            np.zeros((0, INPUT_LEN, len(FEATURE_COLS)), dtype=np.fooat32),
            np.zeros((0, HORIZON), dtype=np.float32)
        )
    return np.concatenate(X_list, axis=0), np.concatenate(y_list, axis=0)


def build_windows_temp_transformer(split_df: pd.DataFrame):
    """Build windows for `TemperatureTransformer` across all `location_id`.

    Returns:
      - z_past: (N, INPUT_LEN, 1)
      - x_cov_past: (N, INPUT_LEN, d_cov) where d_cov=len(FEATURE_COLS)-1
      - x_cov_future: (N, HORIZON, d_cov)
      - y: (N, HORIZON) future temperature target
    """
    z_list, x_cov_past_list, x_cov_future_list, y_list = [], [], [], []

    d_cov = len(FEATURE_COLS) - 1

    for _, grp in split_df.groupby("location_id"):
        grp = grp.sort_values("time")
        mat = grp[FEATURE_COLS].values.astype(np.float32)
        times = grp['time'].values

        z_past, x_cov_past, x_cov_future, y = build_windows_temp_transformer_one_location(
            times, mat, INPUT_LEN, HORIZON, STRIDE
        )

        if z_past is not None:
            z_list.append(z_past)
            x_cov_past_list.append(x_cov_past)
            x_cov_future_list.append(x_cov_future)
            y_list.append(y)

    if not z_list:
        return (
            np.zeros((0, INPUT_LEN, 1), dtype=np.float32),
            np.zeros((0, INPUT_LEN, d_cov), dtype=np.float32),
            np.zeros((0, HORIZON, d_cov), dtype=np.float32),
            np.zeros((0, HORIZON), dtype=np.float32),
        )

    return (
        np.concatenate(z_list, axis=0),
        np.concatenate(x_cov_past_list, axis=0),
        np.concatenate(x_cov_future_list, axis=0),
        np.concatenate(y_list, axis=0),
    )

In [5]:
## Create a boolean fileter to keep only the true rows getting returned
def region_mask(df:pd.DataFrame, region_name: str):
    return df["region"].astype(str).str.strip().str.lower() == region_name.strip().lower()

# Filter Station

In [6]:
def compute_station_stats(papua_train: pd.DataFrame) -> pd.DataFrame:
    if papua_train is None or len(papua_train) == 0:
        return pd.DataFrame(columns=["location_id", "mean_temp", "std_temp", "elevation", "n_rows"])
    df = papua_train.copy()
    df['location_id'] = df['location_id'].astype(str)
    g = df.groupby("location_id", dropna=False)
    
    stats = g.agg(
        mean_temp=(TARGET_COL, "mean"),
        std_temp=(TARGET_COL, "std"),
        elevation=("elevation", "mean"),
        n_rows=(TARGET_COL, "size"),
    ).reset_index()
    
    stats["mean_temp"] = stats["mean_temp"].astype(np.float32)
    stats["std_temp"] = stats["std_temp"].fillna(0.0).astype(np.float32)
    stats["elevation"] = stats["elevation"].astype(np.float32)
    stats["n_rows"] = stats["n_rows"].astype(int)
    return stats

def select_target_stations_papua_by_elevation(papua_train: pd.DataFrame):
    """
    Select station IDs from Papua training set by elevation:
      - Single station: median elevation station
      - Three stations: lowest, median, highest elevation stations
    Returns (single_station_id: str, three_station_ids: list[str], stats_sorted: pd.DataFrame).
    """
    stats = compute_station_stats(papua_train)
    if len(stats) == 0:
        return None, [], stats

    stats_sorted = stats.sort_values(["elevation", "location_id"], ascending=[True, True]).reset_index(drop=True)
    mid = len(stats_sorted) // 2
    single_id = str(stats_sorted.loc[mid, "location_id"])

    low_id = str(stats_sorted.loc[0, "location_id"])
    high_id = str(stats_sorted.loc[len(stats_sorted) - 1, "location_id"])
    three_ids = [low_id, single_id, high_id]

    # If very small number of stations, deduplicate while keeping order
    seen = set()
    three_ids = [x for x in three_ids if not (x in seen or seen.add(x))]
    return single_id, three_ids, stats_sorted


def filter_df_by_station_ids(df: pd.DataFrame, station_ids) -> pd.DataFrame:
    if df is None or len(df) == 0:
        return df.iloc[0:0].copy() if df is not None else df
    ids = set([str(x) for x in station_ids])
    out = df.copy()
    out["location_id"] = out["location_id"].astype(str)
    return out[out["location_id"].isin(ids)]

# Model Architecuture

## Vanilla Transformer

In [8]:
import torch.nn as nn

class VanillaTransformer(nn.Module):
    def __init__(self, input_dim=9, d_model=32, nhead=4, num_layers=2, dropout=0.1, horizon=7):
        """
        Model Parameter:
          - input_dim = 9. Number of feature
          - d_model = 32. Internal Feature Size of transformer. 9 Features -> projected to 32 features
          - nhead = 4. Number of Attention Heads
          - num_layers = 2.  Stack 2 Transformer Layers. Each Layer has (self-attention + Feedforward Network)
          - Horizon = 7. Predict 7 days ahead
        """

        super().__init__()
        ## input Projection. Transform from 9 features to 32 Features
        self.input_proj = nn.Linear(input_dim, d_model)
        # transformer decoder
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model, nhead=nhead, dim_feedforward=d_model*4,
            dropout=dropout, batch_first=True, norm_first=False
        )
        self.encoder = nn.TransformerEncoder(encoder_layer, num_layers=num_layers)
        self.head = nn.Linear(d_model, horizon)

    def forward(self, x):
        # x: (B,T,F)
        x = self.input_proj(x)
        x = self.encoder(x)
        x = x.mean(dim=1)
        return self.head(x)

## Domain Adversarial NN

### Feature Extractor

In [9]:
class FeatureExtractor(nn.Module):
    """Transformer encoder + mean pooling. Output: pooled representation (d_model)."""

    def __init__(self, input_dim: int, d_model: int = 32, nhead: int = 4, num_layers: int = 2, dropout: float = 0.1):
        super().__init__()
        self.input_proj = nn.Linear(input_dim, d_model)
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model,
            nhead=nhead,
            dim_feedforward=d_model * 4,
            dropout=dropout,
            batch_first=True,
            norm_first=False,
        )
        self.encoder = nn.TransformerEncoder(encoder_layer, num_layers=num_layers)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x = self.input_proj(x)
        x = self.encoder(x)
        return x.mean(dim=1)

### Domain Classifier

In [10]:
class DomainClassifier(nn.Module):
    """MLP: Linear(in→32) ReLU Linear(32→1) Sigmoid."""

    def __init__(self, in_dim: int):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(in_dim, 32),
            nn.ReLU(),
            nn.Linear(32, 1),
            nn.Sigmoid(),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.net(x).squeeze(1)  # (B,)


def dann_lambda_schedule(p: float) -> float:
    # λ = 2 / (1 + exp(-10 p)) - 1
    return float(2.0 / (1.0 + math.exp(-10.0 * float(p))) - 1.0)



### Gradient Reversal Layer

In [11]:
class GradReverse(torch.autograd.Function):
    @staticmethod
    def forward(ctx, x: torch.Tensor, lambd: float):
        ctx.lambd = float(lambd)
        return x.view_as(x)

    @staticmethod
    def backward(ctx, grad_output: torch.Tensor):
        return -ctx.lambd * grad_output, None


def grl(x: torch.Tensor, lambd: float) -> torch.Tensor:
    return GradReverse.apply(x, lambd)

## Climate Aware Vanilla Transformer Based

### Non-DA

In [14]:
class ClimateAwareTransformer(nn.Module):
    """
    Temporal local features -> Transformer encoder -> 32-dim pooled
    Geo features (lat, lon, elev) -> Linear(3->16)
    Climate regime (Nino, DMI) -> Linear(2->16)
    Concat -> 64 -> Linear(64->7)
    """

    def __init__(self, horizon: int = 7, d_model: int = 32, nhead: int = 4, num_layers: int = 2, dropout: float = 0.1):
        super().__init__()
        self.idx_temp = [FEATURE_COLS.index(c) for c in LOCAL_TEMPORAL_COLS]
        self.idx_geo = [FEATURE_COLS.index(c) for c in GEO_COLS]
        self.idx_clim = [FEATURE_COLS.index(c) for c in CLIMATE_COLS]

        self.temporal = FeatureExtractor(input_dim=len(self.idx_temp), d_model=d_model, nhead=nhead, num_layers=num_layers, dropout=dropout)
        self.geo_proj = nn.Linear(3, 16)
        self.clim_proj = nn.Linear(2, 16)
        self.head = nn.Linear(d_model + 16 + 16, horizon)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        xt = x[:, :, self.idx_temp]
        rep_t = self.temporal(xt)
        rep_g = self.geo_proj(x[:, 0, self.idx_geo])
        rep_c = self.clim_proj(x[:, 0, self.idx_clim])
        rep = torch.cat([rep_t, rep_g, rep_c], dim=1)
        return self.head(rep)

### DA

In [15]:
class ClimateAwareRep(nn.Module):
    """Same as ClimateAwareTransformer but returns 64-dim representation (for optional DANN)."""

    def __init__(self, d_model: int = 32, nhead: int = 4, num_layers: int = 2, dropout: float = 0.1):
        super().__init__()
        self.idx_temp = [FEATURE_COLS.index(c) for c in LOCAL_TEMPORAL_COLS]
        self.idx_geo = [FEATURE_COLS.index(c) for c in GEO_COLS]
        self.idx_clim = [FEATURE_COLS.index(c) for c in CLIMATE_COLS]

        self.temporal = FeatureExtractor(input_dim=len(self.idx_temp), d_model=d_model, nhead=nhead, num_layers=num_layers, dropout=dropout)
        self.geo_proj = nn.Linear(3, 16)
        self.clim_proj = nn.Linear(2, 16)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        xt = x[:, :, self.idx_temp]
        rep_t = self.temporal(xt)
        rep_g = self.geo_proj(x[:, 0, self.idx_geo])
        rep_c = self.clim_proj(x[:, 0, self.idx_clim])
        return torch.cat([rep_t, rep_g, rep_c], dim=1)  # (B, 64)

## TFT (DA)

### Domain Classifier

In [16]:
class TFTDomainClassifier(nn.Module):
    def __init__(self, hidden_dim):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(hidden_dim, 64),
            nn.ReLU(),
            nn.Linear(64, 2)
        )

    def forward(self, x):
        return self.net(x)

### Gated Residual Network

In [17]:
class TFTGRN(nn.Module):
    def __init__(self, input_dim, hidden_dim, output_dim=None, dropout=0.1):
        super().__init__()
        output_dim = output_dim or input_dim

        self.fc1 = nn.Linear(input_dim, hidden_dim)
        self.fc2 = nn.Linear(hidden_dim, output_dim)
        self.elu = nn.ELU()
        self.dropout = nn.Dropout(dropout)

        self.gate = nn.Linear(output_dim, output_dim)
        self.sigmoid = nn.Sigmoid()

        self.skip = nn.Linear(input_dim, output_dim) if input_dim != output_dim else None
        self.layer_norm = nn.LayerNorm(output_dim)

    def forward(self, x):
        residual = x

        x = self.elu(self.fc1(x))
        x = self.dropout(self.fc2(x))

        gate = self.sigmoid(self.gate(x))
        x = gate * x

        if self.skip is not None:
            residual = self.skip(residual)

        return self.layer_norm(x + residual)

### Variable Selection Network

In [23]:
import torch.nn.functional as F

class TFTVariableSelectionNetwork(nn.Module):
    def __init__(self, input_dim, num_vars, hidden_dim):
        super().__init__()
        self.num_vars = num_vars

        self.var_grns = nn.ModuleList([
            TFTGRN(input_dim, hidden_dim, hidden_dim)
            for _ in range(num_vars)
        ])

        self.weight_grn = TFTGRN(input_dim * num_vars, hidden_dim, num_vars)

    def forward(self, x):
        # x: [B, T, N, D]
        B, T, N, D = x.shape

        var_outputs = []
        for i in range(N):
            var_outputs.append(self.var_grns[i](x[:, :, i, :]))

        var_outputs = torch.stack(var_outputs, dim=2)  # [B, T, N, H]

        flat = x.reshape(B, T, -1)
        weights = F.softmax(self.weight_grn(flat), dim=-1).unsqueeze(-1)

        out = (weights * var_outputs).sum(dim=2)

        return out, weights

### Gradient Reversal Layer

In [19]:
class TFTGRL(torch.autograd.Function):
    @staticmethod
    def forward(ctx, x, lambda_):
        ctx.lambda_ = lambda_
        return x.view_as(x)

    @staticmethod
    def backward(ctx, grad_output):
        return -ctx.lambda_ * grad_output, None

class TFT_DA_Model(nn.Module):
    def __init__(self, tft, hidden_dim):
        super().__init__()
        self.tft = tft

        self.task_head = nn.Linear(hidden_dim, 1)
        self.domain_classifier = TFTDomainClassifier(hidden_dim)

    def forward(self, x, lambda_grl=1.0):
        features, _, _ = self.tft(x)

        # Use last timestep (or pooled)
        feat = features[:, -1, :]

        # Forecast
        y_pred = self.task_head(feat)

        # Domain prediction (with GRL)
        feat_grl = TFTGRL.apply(feat, lambda_grl)
        domain_pred = self.domain_classifier(feat_grl)

        return y_pred, domain_pred

### TFT Backbone/ Feature Extractor

In [22]:
class TFT_Backbone(nn.Module):
    def __init__(
        self,
        input_dim,
        num_vars,
        hidden_dim=64,
        lstm_layers=1,
        num_heads=4,
        dropout=0.1
    ):
        super().__init__()

        self.vsn = TFTVariableSelectionNetwork(input_dim, num_vars, hidden_dim)

        self.lstm = nn.LSTM(
            input_size=hidden_dim,
            hidden_size=hidden_dim,
            num_layers=lstm_layers,
            batch_first=True
        )

        self.attn = nn.MultiheadAttention(
            embed_dim=hidden_dim,
            num_heads=num_heads,
            batch_first=True
        )

        self.post_grn = TFTGRN(hidden_dim, hidden_dim)

    def forward(self, x):
        # x: [B, T, N, D]

        x, var_weights = self.vsn(x)

        lstm_out, _ = self.lstm(x)

        attn_out, attn_weights = self.attn(lstm_out, lstm_out, lstm_out)

        features = self.post_grn(attn_out)

        return features, var_weights, attn_weights

### Full Model

In [21]:
class TFT_DA_Model(nn.Module):
    def __init__(
        self,
        input_dim,
        num_vars,
        hidden_dim=64,
        horizon=7
    ):
        super().__init__()

        self.horizon = horizon

        self.backbone = TFT_Backbone(
            input_dim=input_dim,
            num_vars=num_vars,
            hidden_dim=hidden_dim
        )

        # Forecast head (multi-horizon)
        self.forecast_head = nn.Linear(hidden_dim, horizon)

        # Domain classifier
        self.domain_classifier = TFTDomainClassifier(hidden_dim)

    def forward(self, x, lambda_grl=1.0):
        features, var_w, attn_w = self.backbone(x)

        # Pooling (important!)
        feat = features.mean(dim=1)  # [B, H]

        # Forecast
        forecast = self.forecast_head(feat)  # [B, horizon]

        # Domain prediction with GRL
        feat_grl = TFTGRL.apply(feat, lambda_grl)
        domain_pred = self.domain_classifier(feat_grl)

        return forecast, domain_pred, var_w, attn_w

## TFT2 Non DA

In [30]:
class TemporalAttention(nn.Module):
    def __init__(self, d_model):
        super().__init__()
        self.attn = nn.Linear(d_model, 1)

    def forward(self, x):
        # x: (B, T, d_model)
        scores = self.attn(x)                     # (B, T, 1)
        weights = torch.softmax(scores, dim=1)    # attention over time
        context = (weights * x).sum(dim=1)        # weighted sum
        return context, weights

In [31]:
def quantile_loss(preds, target, quantiles):
    """
    preds: (B, H, Q)
    target: (B, H)
    """
    losses = []
    for i, q in enumerate(quantiles):
        errors = target - preds[:, :, i]
        loss = torch.max((q - 1) * errors, q * errors)
        losses.append(loss.unsqueeze(-1))
    return torch.mean(torch.cat(losses, dim=-1))

In [32]:
class HorizonDecoder(nn.Module):
    def __init__(self, d_model, horizon, num_quantiles):
        super().__init__()
        self.horizon = horizon
        self.query = nn.Parameter(torch.randn(horizon, d_model))

        self.attn = nn.MultiheadAttention(
            embed_dim=d_model,
            num_heads=4,
            batch_first=True
        )

        self.proj = nn.Linear(d_model, num_quantiles)

    def forward(self, context_seq):
        """
        context_seq: (B, T, d_model)
        """
        B = context_seq.size(0)

        # expand learned queries
        q = self.query.unsqueeze(0).expand(B, -1, -1)  # (B, H, d_model)

        # cross-attention: horizon queries attend to time
        out, _ = self.attn(q, context_seq, context_seq)

        return self.proj(out)  # (B, H, Q)

In [33]:
class VariableSelection(nn.Module):
    def __init__(self, num_vars, d_model):
        super().__init__()
        self.weight_net = nn.Sequential(
            nn.Linear(num_vars, num_vars),
            nn.Softmax(dim=-1)
        )
        self.proj = nn.Linear(num_vars, d_model)

    def forward(self, x):
        # x: (B, T, num_vars)
        weights = self.weight_net(x.mean(dim=1))  # (B, num_vars)
        x_weighted = x * weights.unsqueeze(1)
        return self.proj(x_weighted)

In [34]:
class GRN(nn.Module):
    def __init__(self, input_dim, hidden_dim, output_dim=None, dropout=0.1):
        super().__init__()
        output_dim = output_dim or input_dim

        self.fc1 = nn.Linear(input_dim, hidden_dim)
        self.elu = nn.ELU()
        self.fc2 = nn.Linear(hidden_dim, output_dim)

        self.dropout = nn.Dropout(dropout)
        self.gate = nn.Linear(output_dim, output_dim)
        self.norm = nn.LayerNorm(output_dim)

        if input_dim != output_dim:
            self.skip = nn.Linear(input_dim, output_dim)
        else:
            self.skip = None

    def forward(self, x):
        residual = x if self.skip is None else self.skip(x)

        out = self.fc1(x)
        out = self.elu(out)
        out = self.fc2(out)
        out = self.dropout(out)

        gate = torch.sigmoid(self.gate(out))
        out = gate * out + (1 - gate) * residual

        return self.norm(out)

In [35]:
class ClimateTFTv2(nn.Module):
    def __init__(
        self,
        horizon=7,
        d_model=32,
        nhead=4,
        num_layers=2,
        dropout=0.1,
        quantiles=[0.1, 0.5, 0.9],
    ):
        super().__init__()

        self.quantiles = quantiles
        self.num_q = len(quantiles)

        self.idx_temp = [FEATURE_COLS.index(c) for c in LOCAL_TEMPORAL_COLS]
        self.idx_geo = [FEATURE_COLS.index(c) for c in GEO_COLS]
        self.idx_clim = [FEATURE_COLS.index(c) for c in CLIMATE_COLS]

        # --- Variable Selection ---
        self.vsn = VariableSelection(len(self.idx_temp), d_model)

        # --- Transformer ---
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model,
            nhead=nhead,
            dropout=dropout,
            batch_first=True,
        )
        self.transformer = nn.TransformerEncoder(encoder_layer, num_layers=num_layers)

        # --- Attention over time ---
        self.temporal_attn = TemporalAttention(d_model)

        # --- Static encoders ---
        self.geo_grn = GRN(3, 32, d_model)
        self.clim_grn = GRN(2, 32, d_model)

        # --- Fusion ---
        self.fusion_grn = GRN(d_model * 3, 64, d_model)

        # --- Decoder ---
        self.decoder = HorizonDecoder(d_model, horizon, self.num_q)

    def forward(self, x):
        xt = x[:, :, self.idx_temp]
        xg = x[:, 0, self.idx_geo]
        xc = x[:, 0, self.idx_clim]

        # --- Variable selection ---
        xt = self.vsn(xt)

        # --- Transformer ---
        h_seq = self.transformer(xt)

        # --- Temporal attention ---
        h_context, attn_weights = self.temporal_attn(h_seq)

        # --- Static features ---
        g = self.geo_grn(xg)
        c = self.clim_grn(xc)

        # --- Fusion ---
        fused = torch.cat([h_context, g, c], dim=1)
        fused = self.fusion_grn(fused)

        # Expand fused to sequence for decoder
        fused_seq = fused.unsqueeze(1).repeat(1, h_seq.size(1), 1)

        # Combine with temporal sequence
        full_seq = h_seq + fused_seq

        # --- Decoder ---
        out = self.decoder(full_seq)  # (B, H, Q)

        return out, attn_weights

### Gradient Reversal Layer

In [28]:
class GradReverse(torch.autograd.Function):
    @staticmethod
    def forward(ctx, x, lambda_):
        ctx.lambda_ = lambda_
        return x.view_as(x)

    @staticmethod
    def backward(ctx, grad_output):
        return -ctx.lambda_ * grad_output, None


def grad_reverse(x, lambda_=1.0):
    return GradReverse.apply(x, lambda_)

### Domain Classifier

In [29]:
class DomainClassifier(nn.Module):
    def __init__(self, input_dim):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, 64),
            nn.ReLU(),
            nn.Linear(64, 2)  # source vs target
        )

    def forward(self, x):
        return self.net(x)

### Full Model

# Training Configuration

## Vanilla Transformer

### Evals Vanilla Transformer (Supervised)

In [24]:
def eval_model_mae(model: torch.nn.Module, X: np.ndarray, y: np.ndarray, batch_size: int = 256) -> float:
    if X.shape[0] == 0:
        return float("nan")
    model.eval()
    preds = []
    with torch.no_grad():
        for i in range(0, X.shape[0], batch_size):
            xb = torch.from_numpy(X[i : i + batch_size])
            xb = xb.to(DEVICE, non_blocking=(DEVICE.type == "cuda"))
            preds.append(model(xb).detach().cpu().numpy().astype(np.float32))
    pred = np.concatenate(preds, axis=0)
    return float(np.mean(np.abs(pred - y)))

In [ ]:
def eval_model_metrics(model: torch.nn.Module, X: np.ndarray, y: np.ndarray, batch_size: int = 256):
    """
    Compute MAE/MSE/RMSE on window targets (y shape: [N, H]).
    Metrics are computed over all elements (N*H).
    Returns dict: {"mae": float, "mse": float, "rmse": float}.
    """
    if X.shape[0] == 0:
        return {"mae": float("nan"), "mse": float("nan"), "rmse": float("nan")}
    model.eval() # switch model into evaluation mode
    preds = []
    with torch.no_grad(): # disable gradients
        for i in range(0, X.shape[0], batch_size):
            # Extract batch 
            xb = torch.from_numpy(X[i : i + batch_size])
            xb = xb.to(DEVICE, non_blocking=(DEVICE.type == "cuda"))
            preds.append(model(xb).detach().cpu().numpy().astype(np.float32))
    pred = np.concatenate(preds, axis=0).astype(np.float32)
    y = y.astype(np.float32, copy=False)
    err = (pred - y).astype(np.float32)
    mse = float(np.mean(err ** 2))
    mae = float(np.mean(np.abs(err)))
    rmse = float(np.sqrt(mse))
    return {"mae": mae, "mse": mse, "rmse": rmse}



### Train Vanilla Transformer (Supervised)

In [ ]:
def train_supervised_earlystop_target_mae(
    model: nn.Module,
    X_train: np.ndarray,
    y_train: np.ndarray,
    X_tgt_val: np.ndarray,
    y_tgt_val: np.ndarray,
    epochs: int = 100,
    batch_size: int = 256,
    lr: float = 1e-3,
    patience: int = 5,
):
    """Train with MSE; early stop on target validation MAE."""
    from torch.utils.data import TensorDataset, DataLoader

    if X_train.shape[0] == 0:
        return model.to(DEVICE), float("nan")

    model = model.to(DEVICE).float()
    train_ds = TensorDataset(torch.from_numpy(X_train).to(DEVICE), torch.from_numpy(y_train).to(DEVICE))
    train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True, num_workers=0)
    opt = torch.optim.Adam(model.parameters(), lr=lr)

    best_mae, best_state, wait = float("inf"), None, 0

    for ep in range(epochs):
        model.train()
        tr_mse = 0.0
        for xb, yb in train_loader:
            opt.zero_grad() # resetting the gradient from the previous batch. But why?? -> so on the next training iteration the gradient is get set to 0
            pred = model(xb)
            loss = nn.functional.mse_loss(pred, yb)
            loss.backward() # back propagation
            opt.step() # update weights
            tr_mse += loss.item() * xb.size(0) # *** track training loss (accumulates loss accross batches)
        tr_mse /= max(1, X_train.shape[0]) # *** mean training loss per sample

        val_mae = eval_model_mae(model, X_tgt_val, y_tgt_val, batch_size=256)
        print(f"  Epoch {ep+1}/{epochs}  Train MSE: {tr_mse:.6f}  Target Val MAE: {val_mae:.4f}")

        if val_mae < best_mae:
            best_mae = val_mae
            best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
            wait = 0
        # else:
        #     wait += 1
        #     if wait >= patience:
        #         print(f"  Early stopping at epoch {ep+1}")
        #         break

    if best_state is not None:
        model.load_state_dict(best_state)
    return model.to(DEVICE), best_mae


## Domain Adversarial NN

### Train DANN

In [13]:
def train_domain_adversarial_dann(
    feat_extractor: nn.Module,
    task_head: nn.Module,
    domain_clf: nn.Module,
    X_src: np.ndarray,
    y_src: np.ndarray,
    X_tgt: np.ndarray,
    y_tgt: np.ndarray,
    X_tgt_val: np.ndarray,
    y_tgt_val: np.ndarray,
    epochs: int = 25,
    batch_size: int = 256,
    lr: float = 1e-3,
    patience: int = 5,
    use_target_task_loss: bool = False,
):
    """
    DANN:
      loss = forecast_MSE + λ * domain_BCE
      λ schedule: 2/(1+exp(-10p)) - 1, p=step/total_steps
      balanced batches: batch_size/2 source + batch_size/2 target
      early stop: target validation MAE
    """
    from torch.utils.data import TensorDataset, DataLoader

    if X_src.shape[0] == 0 or X_tgt.shape[0] == 0:
        return None, float("nan")

    src_bs = max(1, batch_size // 2)
    tgt_bs = max(1, batch_size // 2)

    src_ds = TensorDataset(torch.from_numpy(X_src).to(DEVICE), torch.from_numpy(y_src).to(DEVICE))
    tgt_ds = TensorDataset(torch.from_numpy(X_tgt).to(DEVICE), torch.from_numpy(y_tgt).to(DEVICE))
    src_loader = DataLoader(src_ds, batch_size=src_bs, shuffle=True, num_workers=0, drop_last=True)
    tgt_loader = DataLoader(tgt_ds, batch_size=tgt_bs, shuffle=True, num_workers=0, drop_last=True)

    feat_extractor = feat_extractor.to(DEVICE).float()
    task_head = task_head.to(DEVICE).float()
    domain_clf = domain_clf.to(DEVICE).float()

    params = list(feat_extractor.parameters()) + list(task_head.parameters()) + list(domain_clf.parameters())
    opt = torch.optim.Adam(params, lr=lr)
    bce = nn.BCELoss()

    total_steps = epochs * min(len(src_loader), len(tgt_loader))
    global_step = 0

    best_mae, best_state, wait = float("inf"), None, 0

    for ep in range(epochs):
        feat_extractor.train()
        task_head.train()
        domain_clf.train()

        tr_task, tr_dom = 0.0, 0.0
        n_steps = min(len(src_loader), len(tgt_loader))
        src_iter = iter(src_loader)
        tgt_iter = iter(tgt_loader)

        for _ in range(n_steps):
            xs, ys = next(src_iter)
            xt, yt = next(tgt_iter)
            xb = torch.cat([xs, xt], dim=0)
            dom_y = torch.cat(
                [torch.zeros(xs.size(0), device=DEVICE), torch.ones(xt.size(0), device=DEVICE)],
                dim=0,
            ).float()

            p = global_step / max(1, total_steps)
            lambd = dann_lambda_schedule(p)
            global_step += 1

            opt.zero_grad()
            rep = feat_extractor(xb)
            yhat = task_head(rep)

            task_loss = nn.functional.mse_loss(yhat[: xs.size(0)], ys)
            if use_target_task_loss:
                task_loss = task_loss + nn.functional.mse_loss(yhat[xs.size(0) :], yt)

            dom_pred = domain_clf(grl(rep, lambd))
            dom_loss = bce(dom_pred, dom_y)

            loss = task_loss + (lambd * dom_loss)
            loss.backward()
            opt.step()

            tr_task += float(task_loss.item())
            tr_dom += float(dom_loss.item())

        eval_model = nn.Sequential(feat_extractor, task_head)
        val_mae = eval_model_mae(eval_model, X_tgt_val, y_tgt_val, batch_size=256)
        tr_task /= max(1, n_steps)
        tr_dom /= max(1, n_steps)
        print(f"  Epoch {ep+1}/{epochs}  Task MSE: {tr_task:.6f}  Domain BCE: {tr_dom:.6f}  Target Val MAE: {val_mae:.4f}")

        if val_mae < best_mae:
            best_mae = val_mae
            best_state = {
                "feat": {k: v.cpu().clone() for k, v in feat_extractor.state_dict().items()},
                "task": {k: v.cpu().clone() for k, v in task_head.state_dict().items()},
                "dom": {k: v.cpu().clone() for k, v in domain_clf.state_dict().items()},
            }
            wait = 0
        else:
            wait += 1
            if wait >= patience:
                print(f"  Early stopping at epoch {ep+1}")
                break

    if best_state is not None:
        feat_extractor.load_state_dict(best_state["feat"])
        task_head.load_state_dict(best_state["task"])
        domain_clf.load_state_dict(best_state["dom"])

    return nn.Sequential(feat_extractor, task_head).to(DEVICE), best_mae



## TFT (DA)

### Evaluation Eval TFT

In [26]:
def eval_tft_model_mae(model: torch.nn.Module, X: np.ndarray, y: np.ndarray, batch_size: int = 256) -> float:
    if X.shape[0] == 0:
        return float("nan")

    model.eval()
    preds = []

    with torch.no_grad():
        for i in range(0, X.shape[0], batch_size):
            xb = torch.from_numpy(X[i : i + batch_size])
            xb = xb.to(DEVICE, non_blocking=(DEVICE.type == "cuda"))

            # 🔥 FIX HERE
            yhat, _, _, _ = model(xb, lambda_grl=0.0)

            preds.append(yhat.cpu().numpy().astype(np.float32))

    pred = np.concatenate(preds, axis=0)
    return float(np.mean(np.abs(pred - y)))

### Eval Metrics TFT

In [27]:
def tft_eval_model_metrics(model, X, y, batch_size=256):
    model.eval()
    preds = []

    with torch.no_grad():
        for i in range(0, len(X), batch_size):
            xb = torch.from_numpy(X[i:i+batch_size]).to(DEVICE)
            yhat, _, _, _ = model(xb, lambda_grl=0.0)
            preds.append(yhat.cpu())

    preds = torch.cat(preds, dim=0)
    y = torch.from_numpy(y)

    mae = torch.mean(torch.abs(preds - y)).item()
    mse = torch.mean((preds - y) ** 2).item()
    rmse = mse ** 0.5

    return {"mae": mae, "mse": mse, "rmse": rmse}

## Spacial Scarcity EXP

In [ ]:
def run_target_station_experiment(
    experiment_name: str,
    train_java_df: pd.DataFrame,
    train_papua_df: pd.DataFrame,
    val_papua_df: pd.DataFrame,
    test_papua_df: pd.DataFrame,
    beta: np.ndarray,
    target_station_ids,
    epochs: int = 100,
    batch_size: int = 256,
    lr: float = 1e-3
):
    """
    Train multiple models for a given target-station subset and evaluate on FULL Papua test set:
      - Vanilla Transformer (Papua-only, supervised)
      - DANN (Java vs selected Papua)
      - KMM (Java reweighted to selected Papua)
      - Climate-Aware Transformer (Papua-only, supervised)
      - Climate-Aware + DANN (Java vs selected Papua)
    Optionally includes a precomputed ARIMA baseline dict.
    """
    print("\n" + "=" * 60)
    print("Retrain with no early stop/ 100 epoch/ batch_size 256")
    print(f"{experiment_name}")
    print("=" * 60)
    print(f"Selected Papua target stations (train only): {', '.join([str(x) for x in target_station_ids])}")

    train_papua_sel = filter_df_by_station_ids(train_papua_df, target_station_ids)

    X_src, y_src = build_windows(train_java_df)
    X_tgt, y_tgt = build_windows(train_papua_sel)
    X_tgt_val, y_tgt_val = build_windows(val_papua_df)  # FULL Papua val
    X_tgt_test, y_tgt_test = build_windows(test_papua_df)  # FULL Papua test

    print(f"Java train windows:         {X_src.shape[0]}")
    print(f"Papua train windows (sel):  {X_tgt.shape[0]}")
    print(f"Papua val windows (FULL):   {X_tgt_val.shape[0]}")
    print(f"Papua test windows (FULL):  {X_tgt_test.shape[0]}")

    if X_tgt.shape[0] == 0 or X_tgt_val.shape[0] == 0 or X_tgt_test.shape[0] == 0:
        print("Not enough windows for this experiment; skipping.")
        out = {
           # "arima": (arima_baseline or {"mae": float("nan"), "mse": float("nan"), "rmse": float("nan")}),
            "vanilla": {"mae": float("nan"), "mse": float("nan"), "rmse": float("nan")},
            "dann": {"mae": float("nan"), "mse": float("nan"), "rmse": float("nan")},
            "kmm": {"mae": float("nan"), "mse": float("nan"), "rmse": float("nan")},
            #"climate_aware": {"mae": float("nan"), "mse": float("nan"), "rmse": float("nan")},
            "climate_aware_dann": {"mae": float("nan"), "mse": float("nan"), "rmse": float("nan")},
        }
        return out

    # -----------------------------
    # A) Vanilla: Papua-only (no Java)
    # -----------------------------
    print("\n" + "-" * 60)
    print("A) Vanilla Transformer (Papua only)")
    print("-" * 60)
    vanilla = VanillaTransformer(
        input_dim=X_tgt.shape[2],
        d_model=32,
        nhead=4,
        num_layers=2,
        dropout=0.1,
        horizon=HORIZON,
    )
    vanilla, best_val_mae = train_supervised_earlystop_target_mae(
        vanilla,
        X_tgt,
        y_tgt,
        X_tgt_val,
        y_tgt_val,
        epochs=epochs,
        batch_size=batch_size,
        lr=lr,
        patience=patience,
    )
    vanilla_metrics = eval_model_metrics(vanilla, X_tgt_test, y_tgt_test, batch_size=256)
    print(f"Best Papua Val MAE (early stop): {best_val_mae:.4f}")
    print(
        f"Papua Test  MAE: {vanilla_metrics['mae']:.4f}  "
        f"RMSE: {vanilla_metrics['rmse']:.4f}  MSE: {vanilla_metrics['mse']:.6f}"
    )

    # -----------------------------
    # B) DANN: adversarial DA
    # -----------------------------
    dann_metrics = {"mae": float("nan"), "mse": float("nan"), "rmse": float("nan")}
    if X_src.shape[0] > 0:
        print("\n" + "-" * 60)
        print("B) DANN (Java vs selected Papua; balanced batches)")
        print("-" * 60)
        feat = FeatureExtractor(input_dim=X_src.shape[2], d_model=32, nhead=4, num_layers=3, dropout=0.1)
        task_head = nn.Linear(32, HORIZON)
        dom = DomainClassifier(in_dim=32)
        dann_model, best_val_mae_dann = train_domain_adversarial_dann(
            feat,
            task_head,
            dom,
            X_src,
            y_src,
            X_tgt,
            y_tgt,
            X_tgt_val,
            y_tgt_val,
            epochs=epochs,
            batch_size=batch_size,
            lr=lr,
            patience=patience,
            use_target_task_loss=False,
        )
        dann_metrics = eval_model_metrics(dann_model, X_tgt_test, y_tgt_test, batch_size=256)
        print(f"Best Papua Val MAE (early stop): {best_val_mae_dann:.4f}")
        print(
            f"Papua Test  MAE: {dann_metrics['mae']:.4f}  "
            f"RMSE: {dann_metrics['rmse']:.4f}  MSE: {dann_metrics['mse']:.6f}"
        )
    else:
        print("\n" + "-" * 60)
        print("B) DANN skipped (no Java/source windows)")
        print("-" * 60)

    # -----------------------------
    # C) KMM: reweight Java -> selected Papua
    # -----------------------------
    kmm_metrics = {"mae": float("nan"), "mse": float("nan"), "rmse": float("nan")}
    if X_src.shape[0] > 0 and X_tgt.shape[0] > 0:
        print("\n" + "-" * 60)
        print("C) KMM (Java reweighted to selected Papua)")
        print("-" * 60)
        try:
            X_src_flat = X_src.reshape(X_src.shape[0], -1)
            X_tgt_flat = X_tgt.reshape(X_tgt.shape[0], -1)
            subsample_src = min(2000, X_src_flat.shape[0])
            subsample_tgt = min(800, X_tgt_flat.shape[0])
            rs_s = np.random.RandomState(42)
            rs_t = np.random.RandomState(43)
            idx_s = rs_s.choice(X_src_flat.shape[0], subsample_src, replace=False)
            idx_t = rs_t.choice(X_tgt_flat.shape[0], subsample_tgt, replace=False)
            # beta = kmm_weights(X_src_flat[idx_s], X_tgt_flat[idx_t], B=2.0, eps=0.1)
            kmm_model, best_val_loss = train_transformer_weighted(
                X_src[idx_s],
                y_src[idx_s],
                X_tgt_val,
                y_tgt_val,
                beta,
                epochs=max(5, min(epochs, 25)),
                batch_size=batch_size,
                lr=lr,
                patience=patience,
            )
            kmm_metrics = eval_model_metrics(kmm_model, X_tgt_test, y_tgt_test, batch_size=256)
            print(f"Best Val loss (early stop): {best_val_loss:.6f}")
            print(
                f"Papua Test  MAE: {kmm_metrics['mae']:.4f}  "
                f"RMSE: {kmm_metrics['rmse']:.4f}  MSE: {kmm_metrics['mse']:.6f}"
            )
        except Exception as e:
            print(f"KMM skipped: {e}")
    else:
        print("\n" + "-" * 60)
        print("C) KMM skipped (no Java/source or no target windows)")
        print("-" * 60)

    # -----------------------------
    # D) Climate-Aware: Papua-only supervised
    # -----------------------------
    print("\n" + "-" * 60)
    print("D) Climate-Aware Transformer (Papua only)")
    print("-" * 60)
    climate_model = ClimateAwareTransformer(horizon=HORIZON, d_model=32, nhead=4, num_layers=2, dropout=0.1)
    climate_model, best_val_mae_clim = train_supervised_earlystop_target_mae(
        climate_model,
        X_tgt,
        y_tgt,
        X_tgt_val,
        y_tgt_val,
        epochs=epochs,
        batch_size=batch_size,
        lr=lr,
        patience=patience,
    )
    climate_metrics = eval_model_metrics(climate_model, X_tgt_test, y_tgt_test, batch_size=256)
    print(f"Best Papua Val MAE (early stop): {best_val_mae_clim:.4f}")
    print(
        f"Papua Test  MAE: {climate_metrics['mae']:.4f}  "
        f"RMSE: {climate_metrics['rmse']:.4f}  MSE: {climate_metrics['mse']:.6f}"
    )

    # -----------------------------
    # E) Climate-Aware + DANN
    # -----------------------------
    ca_dann_metrics = {"mae": float("nan"), "mse": float("nan"), "rmse": float("nan")}
    if X_src.shape[0] > 0:
        print("\n" + "-" * 60)
        print("E) Climate-Aware + DANN (Java vs selected Papua)")
        print("-" * 60)
        ca_feat = ClimateAwareRep(d_model=32, nhead=4, num_layers=2, dropout=0.1)
        ca_task = nn.Linear(64, HORIZON)
        ca_dom = DomainClassifier(in_dim=64)
        ca_dann_model, best_val_mae_cadann = train_domain_adversarial_dann(
            ca_feat,
            ca_task,
            ca_dom,
            X_src,
            y_src,
            X_tgt,
            y_tgt,
            X_tgt_val,
            y_tgt_val,
            epochs=epochs,
            batch_size=batch_size,
            lr=lr,
            patience=patience,
            use_target_task_loss=False,
        )
        ca_dann_metrics = eval_model_metrics(ca_dann_model, X_tgt_test, y_tgt_test, batch_size=256)
        print(f"Best Papua Val MAE (early stop): {best_val_mae_cadann:.4f}")
        print(
            f"Papua Test  MAE: {ca_dann_metrics['mae']:.4f}  "
            f"RMSE: {ca_dann_metrics['rmse']:.4f}  MSE: {ca_dann_metrics['mse']:.6f}"
        )
    else:
        print("\n" + "-" * 60)
        print("E) Climate-Aware + DANN skipped (no Java/source windows)")
        print("-" * 60)

    out = {
        "vanilla": vanilla_metrics,
        "dann": dann_metrics,
        "kmm": kmm_metrics,
        "climate_aware_dann": ca_dann_metrics,
    }

    print("\n" + "-" * 60)
    print("Result (Papua test):")
    for k in ["vanilla", "dann", "kmm", "climate_aware_dann"]:
        m = out[k]
        print(f"  {k:>18s}  MAE={m['mae']:.4f}  MSE={m['mse']:.6f}  RMSE={m['rmse']:.4f}")
    return out

